# Kaggle ETL：电商用户行为数据

分块 ETL 版本：

每个 chunk 完成清洗→构建→写出→释放内存，适配当前数据模型（`dim_time`、`dim_users`、`dim_products`、`fact_user_behavior`）。

## 1. 加载配置与分块参数

In [ ]:
import gc
from pathlib import Path
import pandas as pd

# Kaggle 输入/输出目录
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_OUTPUT = Path("/kaggle/working")

# 分块大小（按机器内存调整）
CHUNK_SIZE = 1000000

# 分块暂存与最终输出目录
STAGING_DIR = KAGGLE_OUTPUT / "staging"
OUTPUT_DIR = KAGGLE_OUTPUT / "outputs"

# 事件日志路径（Kaggle 数据集）
EVENTS_CSV = Path(
    "/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv"
)

if KAGGLE_INPUT.exists():
    print("Kaggle input root:")
    for p in KAGGLE_INPUT.iterdir():
        print("-", p)
else:
    print("Not running on Kaggle. Please set input paths manually.")

## 2. 分块读取与清洗：函数准备

In [ ]:
def ensure_columns(df: pd.DataFrame, defaults: dict) -> pd.DataFrame:
    for col, default_value in defaults.items():
        if col not in df.columns:
            df[col] = default_value
    return df


def normalize_events(df: pd.DataFrame) -> pd.DataFrame:
    column_map = {
        "event_time": "event_time",
        "event_type": "event_type",
        "product_id": "product_id",
        "category_id": "category_id",
        "category_code": "category_code",
        "brand": "brand",
        "price": "price",
        "user_id": "user_id",
        "user_session": "user_session",
    }

    df = df.rename(columns=column_map)
    df = ensure_columns(
        df,
        {
            "category_code": "unknown",
            "brand": "Unknown",
            "price": 0.0,
            "user_session": "unknown",
        },
    )

    df["category_code"] = df["category_code"].fillna("unknown")
    df["brand"] = df["brand"].fillna("Unknown")
    df["user_session"] = df["user_session"].fillna("unknown")

    category_split = df["category_code"].astype(str).str.split(".", expand=True)
    df["category_l1"] = category_split[0].fillna("unknown")
    df["category_l2"] = category_split[1].fillna("other")
    df["category_l3"] = category_split[2].fillna("other")

    return df


def clean_events(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce", utc=True)
    df["event_type"] = df["event_type"].astype(str).str.lower().str.strip()
    df["price"] = pd.to_numeric(df["price"], errors="coerce").fillna(0)

    df = df.drop_duplicates()
    df = df[df["price"] >= 0]
    df = df[df["user_id"].notna()]
    df = df[df["product_id"].notna()]

    return df

## 3. 分块 ETL：清洗→构建→写出→释放内存

In [ ]:
def price_bucket(value: float) -> str:
    if value < 50:
        return "low"
    if value < 200:
        return "mid"
    return "high"


if not EVENTS_CSV.exists():
    print(f"Missing: {EVENTS_CSV}")
else:
    STAGING_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    user_stats: dict[int, tuple[pd.Timestamp, pd.Timestamp]] = {}
    product_stats: dict[tuple, tuple[float, int]] = {}
    min_date = None
    max_date = None
    fact_stage_paths: list[Path] = []

    for idx, chunk in enumerate(pd.read_csv(EVENTS_CSV, chunksize=CHUNK_SIZE)):
        chunk = normalize_events(chunk)
        chunk = clean_events(chunk)

        chunk["event_date"] = pd.to_datetime(chunk["event_time"], utc=True).dt.date
        chunk["time_key"] = (
            chunk["event_date"].astype(str).str.replace("-", "").astype(int)
        )

        chunk_min = chunk["event_date"].min()
        chunk_max = chunk["event_date"].max()
        if min_date is None or chunk_min < min_date:
            min_date = chunk_min
        if max_date is None or chunk_max > max_date:
            max_date = chunk_max

        user_chunk = chunk.groupby("user_id", as_index=False).agg(
            first_seen_date=("event_date", "min"),
            last_seen_date=("event_date", "max"),
        )
        for row in user_chunk.itertuples(index=False):
            existing = user_stats.get(row.user_id)
            if existing is None:
                user_stats[row.user_id] = (row.first_seen_date, row.last_seen_date)
            else:
                user_stats[row.user_id] = (
                    min(existing[0], row.first_seen_date),
                    max(existing[1], row.last_seen_date),
                )

        product_chunk = chunk.groupby(
            [
                "product_id",
                "category_id",
                "category_l1",
                "category_l2",
                "category_l3",
                "brand",
            ],
            as_index=False,
        ).agg(price_sum=("price", "sum"), price_count=("price", "size"))
        for row in product_chunk.itertuples(index=False):
            key = (
                row.product_id,
                row.category_id,
                row.category_l1,
                row.category_l2,
                row.category_l3,
                row.brand,
            )
            existing_sum, existing_count = product_stats.get(key, (0.0, 0))
            product_stats[key] = (
                existing_sum + row.price_sum,
                existing_count + row.price_count,
            )

        fact_stage = chunk[
            [
                "time_key",
                "user_id",
                "product_id",
                "event_type",
                "price",
                "user_session",
            ]
        ].copy()
        fact_stage_path = STAGING_DIR / f"fact_stage_{idx:04d}.parquet"
        fact_stage.to_parquet(fact_stage_path, index=False)
        fact_stage_paths.append(fact_stage_path)

        print(f"Chunk {idx} staged: {len(fact_stage)} rows")
        del chunk, user_chunk, product_chunk, fact_stage
        gc.collect()

    print(f"Staged {len(fact_stage_paths)} chunks.")

## 4. 维表构建（汇总所有 chunk）

In [ ]:
dim_users = None
dim_products = None
dim_time = None

if "user_stats" in globals() and user_stats:
    dim_users = pd.DataFrame(
        [
            {
                "user_id": user_id,
                "first_seen_date": stats[0],
                "last_seen_date": stats[1],
            }
            for user_id, stats in user_stats.items()
        ]
    )
    dim_users = dim_users.sort_values("user_id").reset_index(drop=True)
    dim_users["user_segment"] = "new"
    dim_users["region"] = "unknown"
    dim_users["device_type"] = "desktop"
    dim_users["user_key"] = dim_users.index + 1
    dim_users = dim_users[
        [
            "user_key",
            "user_id",
            "first_seen_date",
            "last_seen_date",
            "user_segment",
            "region",
            "device_type",
        ]
    ]
    dim_users.to_parquet(OUTPUT_DIR / "dim_users.parquet", index=False)
    print("dim_users saved:", dim_users.shape)

if "product_stats" in globals() and product_stats:
    dim_products = pd.DataFrame(
        [
            {
                "product_id": key[0],
                "category_id": key[1],
                "category_l1": key[2],
                "category_l2": key[3],
                "category_l3": key[4],
                "brand": key[5],
                "price": (stats[0] / stats[1]) if stats[1] else 0.0,
            }
            for key, stats in product_stats.items()
        ]
    )
    dim_products["price_range"] = dim_products["price"].apply(price_bucket)
    dim_products = dim_products.sort_values("product_id").reset_index(drop=True)
    dim_products["product_key"] = dim_products.index + 1
    dim_products = dim_products[
        [
            "product_key",
            "product_id",
            "category_id",
            "category_l1",
            "category_l2",
            "category_l3",
            "brand",
            "price_range",
        ]
    ]
    dim_products.to_parquet(OUTPUT_DIR / "dim_products.parquet", index=False)
    print("dim_products saved:", dim_products.shape)

if min_date is not None and max_date is not None:
    date_range = pd.date_range(min_date, max_date, freq="D")
    dim_time = pd.DataFrame({"date_actual": date_range.date})
    dim_time["time_key"] = (
        dim_time["date_actual"].astype(str).str.replace("-", "").astype(int)
    )
    dim_time["year"] = date_range.year
    dim_time["quarter"] = date_range.quarter
    dim_time["month"] = date_range.month
    dim_time["week"] = date_range.isocalendar().week.astype(int)
    dim_time["day_of_week"] = date_range.dayofweek
    dim_time["is_weekend"] = dim_time["day_of_week"].isin([5, 6])
    dim_time["is_holiday"] = False
    dim_time = dim_time[
        [
            "time_key",
            "date_actual",
            "year",
            "quarter",
            "month",
            "week",
            "day_of_week",
            "is_weekend",
            "is_holiday",
        ]
    ]
    dim_time.to_parquet(OUTPUT_DIR / "dim_time.parquet", index=False)
    print("dim_time saved:", dim_time.shape)

## 5. 事实表构建（从分块暂存生成最终输出）

In [ ]:
fact_output_dir = OUTPUT_DIR / "fact_user_behavior_parts"
fact_output_dir.mkdir(parents=True, exist_ok=True)

if dim_users is None or dim_products is None:
    print("dim_users or dim_products missing. Please run previous cells.")
else:
    stage_paths = sorted(STAGING_DIR.glob("fact_stage_*.parquet"))
    for idx, part_path in enumerate(stage_paths):
        fact = pd.read_parquet(part_path)
        fact = fact[
            [
                "time_key",
                "user_id",
                "product_id",
                "event_type",
                "price",
                "user_session",
            ]
        ].copy()
        fact["quantity"] = 1
        fact["revenue"] = fact["price"] * fact["quantity"]

        out_path = fact_output_dir / f"fact_user_behavior_{idx:04d}.parquet"
        fact.to_parquet(out_path, index=False)
        print("fact saved:", out_path.name, len(fact))
        del fact
        gc.collect()

## 6. 特征派生：用户日粒度聚合（分块汇总）

In [ ]:
user_daily_features = None
feature_parts: list[pd.DataFrame] = []

if fact_output_dir.exists():
    for part_path in sorted(fact_output_dir.glob("fact_user_behavior_*.parquet")):
        part_df = pd.read_parquet(part_path)
        features = part_df.groupby(["user_id", "time_key"], as_index=False).agg(
            event_count=("event_type", "count"),
            purchase_count=("event_type", lambda s: (s == "purchase").sum()),
            gmv=("revenue", "sum"),
        )
        feature_parts.append(features)
        del part_df, features
        gc.collect()

    if feature_parts:
        user_daily_features = (
            pd.concat(feature_parts, ignore_index=True)
            .groupby(["user_id", "time_key"], as_index=False)
            .agg(
                event_count=("event_count", "sum"),
                purchase_count=("purchase_count", "sum"),
                gmv=("gmv", "sum"),
            )
        )
        user_daily_features.to_parquet(
            OUTPUT_DIR / "user_daily_features.parquet", index=False
        )
        print("user_daily_features saved:", user_daily_features.shape)

## 7. 合并产出：宽表/训练表（抽样生成）

In [ ]:
wide_table_sample = None
sample_rows = 200000

if dim_users is not None and dim_products is not None and dim_time is not None:
    first_part = next(
        iter(sorted(fact_output_dir.glob("fact_user_behavior_*.parquet"))), None
)
    if first_part is not None:
        fact_sample = pd.read_parquet(first_part).head(sample_rows)
        wide_table_sample = fact_sample.merge(dim_users, on="user_id", how="left")
        wide_table_sample = wide_table_sample.merge(
            dim_products, on="product_id", how="left"
        )
        wide_table_sample = wide_table_sample.merge(dim_time, on="time_key", how="left")
        print("wide_table_sample:", wide_table_sample.shape)

## 8. 导出结果到 Kaggle 输出目录

In [ ]:
print("Outputs:")
print("- dim_time.parquet", OUTPUT_DIR / "dim_time.parquet")
print("- dim_users.parquet", OUTPUT_DIR / "dim_users.parquet")
print("- dim_products.parquet", OUTPUT_DIR / "dim_products.parquet")
print("- fact parts", OUTPUT_DIR / "fact_user_behavior_parts")
print("- user_daily_features.parquet", OUTPUT_DIR / "user_daily_features.parquet")
print("Staging:", STAGING_DIR)

## 9. 本地导入提示（从 Kaggle 输出到 Supabase）

Kaggle 无法稳定直连外部数据库，建议先导出 Parquet，再在本地执行导入脚本。

In [ ]:
print("Local import example:")
print("python etl/local_import.py --data-dir D:/kaggle_output/outputs --truncate")
print("Set DATABASE_URL in your local environment before running.")

## 10. 数据质量校验（抽样/维表）

In [ ]:
def data_quality_report(df: pd.DataFrame, name: str) -> None:
    print(f"{name} rows: {len(df)}")
    print(f"{name} duplicate rows: {df.duplicated().sum()}")
    print(
        f"{name} missing rate:\n{df.isna().mean().sort_values(ascending=False).head(10)}"
    )


if dim_users is not None:
    data_quality_report(dim_users, "dim_users")
if dim_products is not None:
    data_quality_report(dim_products, "dim_products")
if dim_time is not None:
    data_quality_report(dim_time, "dim_time")

## 11. 说明：分块事实表为多文件输出

In [ ]:
print("fact_user_behavior 采用分块输出，文件位于:")
print(fact_output_dir)
print("如需合并，可在本地使用 pandas.concat 或 parquet dataset 进行加载。")

## 12. 将需要的文件打包成压缩包

In [ ]:
import os
import zipfile

# 进入工作目录
os.chdir("/kaggle/working/")

# 定义要打包的目录和压缩包名
target_dir = "outputs"
zip_filename = "only_outputs.zip"

# 打包整个 outputs 文件夹
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(target_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, target_dir)
            zipf.write(file_path, arcname)

print(f"已打包完成：{zip_filename}")